# CDC Cache - Change Data Capture

CDC (Change Data Capture) is a pattern used to track and process only the changes (inserts, updates, deletes) in your data, rather than reprocessing the entire dataset every time.

The `CDCCache` class in Flypipe extends the regular `Cache` functionality with additional methods to track and filter data based on timestamps. This enables incremental processing where only new or changed data flows through your pipeline.

## Key Concepts

- **`read_cdc()`**: Filters incoming data to only include rows that have been updated since the last processing
- **`write_cdc()`**: Records metadata about when data was processed, enabling future filtering
- **`cdc_datetime_updated`**: Timestamp column tracking when each row was last processed
- **Incremental Processing**: Process only new/changed rows and append to existing results

## When to Use CDC Cache

- **Large datasets**: When reprocessing all data is expensive
- **Incremental loads**: When source data arrives in batches over time
- **Data warehousing**: Building fact tables with regular updates
- **Real-time pipelines**: Processing streaming data in micro-batches


## Implementing CDC Cache

To implement CDC cache, you need to:

1. Create a `CDCManager` class to track CDC metadata (timestamps, source/destination nodes)
2. Create a cache class that extends `CDCCache` and implements:
   - Standard cache methods: `read()`, `write()`, `exists()`
   - CDC-specific methods: `read_cdc()`, `write_cdc()`
   - The `CDCManager` is created internally within the cache class
   - Specify `merge_keys` for Delta Lake MERGE INTO operations
3. Use `CacheMode.MERGE` to trigger upsert operations on cached data


### Step 1: CDC Manager with Spark Table

The `CDCManager` tracks when each edge (source → destination) in the pipeline was last processed.


In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, TimestampType


class CDCManager:
    """Manager that tracks CDC metadata using a Spark table"""
    
    def __init__(self, spark, table="cdc_metadata", schema="default", catalog=None):
        self.spark = spark
        self.table = table
        self.schema = schema
        self.catalog = catalog
        self._ensure_table_exists()
    
    @property
    def full_table_name(self):
        """Returns the fully qualified table name"""
        if self.catalog:
            return f"{self.catalog}.{self.schema}.{self.table}"
        return f"{self.schema}.{self.table}"
    
    def _ensure_table_exists(self):
        """Create CDC metadata table if it doesn't exist"""
        cdc_schema = StructType([
            StructField("source", StringType(), False),
            StructField("destiny", StringType(), False),
            StructField("cdc_datetime_updated", TimestampType(), False)
        ])
        
        # Create database if it doesn't exist
        if not self.spark.catalog.databaseExists(self.schema):
            self.spark.sql(f"CREATE DATABASE IF NOT EXISTS {self.schema}")
        
        # Create empty table if it doesn't exist
        if not self.spark.catalog.tableExists(self.table, self.schema):
            empty_df = self.spark.createDataFrame([], cdc_schema)
            empty_df.write.format("delta").mode("overwrite").saveAsTable(self.full_table_name)
            print(f"Created CDC metadata table: {self.full_table_name}")
    
    def write(self, spark, current_node, upstream_nodes, timestamp):
        """Write CDC timestamp entries for current_node and its upstream nodes"""
        from pyspark.sql.functions import lit
        
        # Create entries for each upstream dependency
        entries = []
        for source_node in upstream_nodes:
            entries.append((source_node.key, current_node.key, timestamp))
        
        if entries:
            cdc_schema = StructType([
                StructField("source", StringType(), False),
                StructField("destiny", StringType(), False),
                StructField("cdc_datetime_updated", TimestampType(), False)
            ])
            new_entries_df = spark.createDataFrame(entries, cdc_schema)
            
            # Append to CDC metadata table
            new_entries_df.write.format("delta").mode("append").saveAsTable(
                self.full_table_name, 
                mergeSchema=True
            )
    
    def filter(self, spark, from_node, to_node, df):
        """
        Filter dataframe to return only rows with cdc_datetime_updated after last processed timestamp.
        """
        # Get the last processed timestamp for this edge
        edge_query = f"""
            SELECT MAX(cdc_datetime_updated) as last_timestamp
            FROM {self.full_table_name}
            WHERE source = '{from_node.key}' AND destiny = '{to_node.key}'
        """
        
        last_timestamp_df = spark.sql(edge_query)
        last_timestamp = last_timestamp_df.collect()[0]["last_timestamp"]
        
        if last_timestamp is None:
            # No previous run, return all data
            print(f"  → No CDC history found for {from_node.key} → {to_node.key}, processing all data")
            return df
        
        # Filter dataframe to return only rows updated after last timestamp
        if "cdc_datetime_updated" in df.columns:
            filtered_df = df.filter(df["cdc_datetime_updated"] > last_timestamp)
            row_count = filtered_df.count()
            print(f"  → CDC filter: {row_count} new rows since {last_timestamp}")
            return filtered_df
        
        # If no cdc_datetime_updated column, return all data
        return df


### Step 2: CDC Cache Implementation

This cache implementation uses Delta Lake tables for storage and integrates with the CDC manager for timestamp tracking. It uses Delta Lake's MERGE INTO for upsert operations based on merge keys.


In [ ]:
from flypipe.cache.cdc_cache import CDCCache


class IncrementalCDCCache(CDCCache):
    """CDC Cache that supports incremental processing with Spark tables"""
    
    def __init__(self, spark, table, merge_keys, schema="default", catalog=None, cdc_table="cdc_metadata"):
        self.table = table
        self.merge_keys = merge_keys
        self.schema = schema
        self.catalog = catalog
        self.cdc_table = cdc_table
        self.cdc_manager = CDCManager(spark)
    
    @property
    def full_table_name(self):
        """Returns the fully qualified table name"""
        if self.catalog:
            return f"{self.catalog}.{self.schema}.{self.table}"
        return f"{self.schema}.{self.table}"
    
    def read(self, spark):
        """Read cached data from Delta table"""
        if not self.exists(spark):
            return None
        return spark.table(self.full_table_name)
    
    def write(self, spark, df):
        """
        Write cache - merge into existing Delta table using MERGE INTO.
        This simulates incremental table updates with upsert logic.
        """
        if not spark.catalog.databaseExists(self.schema):
            spark.sql(f"CREATE DATABASE IF NOT EXISTS {self.schema}")
        
        if not self.exists(spark):
            # First write - create table
            print(f"  → Creating table {self.full_table_name}")
            df.write.format("delta").mode("overwrite").saveAsTable(self.full_table_name)
        else:
            # Subsequent writes - merge into existing table
            print(f"  → Merging into table {self.full_table_name}")
            df.createOrReplaceTempView("updates")
            
            # Build merge condition based on merge keys
            merge_condition = " AND ".join([f"target.{key} = source.{key}" for key in self.merge_keys])
            
            merge_query = f"""
                MERGE INTO {self.full_table_name} AS target
                USING updates AS source
                ON {merge_condition}
                WHEN MATCHED THEN UPDATE SET *
                WHEN NOT MATCHED THEN INSERT *
            """
            
            spark.sql(merge_query)
    
    def exists(self, spark):
        return spark.catalog.tableExists(self.table, self.schema)
    
    def read_cdc(self, spark, from_node, to_node, df):
        """
        Filter incoming data using CDC manager.
        Returns only rows that should be processed based on timestamp tracking.
        """
        return self.cdc_manager.filter(spark, from_node, to_node, df)
    
    def write_cdc(self, spark, current_node, upstream_nodes, timestamp):
        """Write CDC metadata tracking when data was processed"""
        self.cdc_manager.write(spark, current_node, upstream_nodes, timestamp)


### Step 3: Define the Pipeline with CDC Cache

We'll create a simple pipeline:

```
    t1 (input passthrough, no cache)
     |
     v
    t2 (transformation with CDC cache)
     |
     v
    t3 (final node with CDC cache)
```

Each transformation adds a `cdc_datetime_updated` timestamp to track when rows were processed.


In [ ]:
from flypipe import node
from pyspark.sql.functions import current_timestamp


@node(type="pyspark")
def t1():
    """Source node - passthrough for input data"""
    return spark.createDataFrame(
        data=[(1, "Alice"), (2, "Bob")],
        schema=["id", "name"]
    )


@node(
    type="pyspark",
    dependencies=[t1],
    cache=IncrementalCDCCache(
        spark=spark,
        table="customer_data_processed",
        merge_keys=["id"],
        schema="default",
        cdc_table="cdc_metadata"
    )
)
def t2(t1):
    """
    Transformation node with CDC cache.
    Adds processing logic and timestamp.
    """
    return t1.selectExpr(
        "id",
        "name",
        "CONCAT(name, '_processed') as processed_name"
    ).withColumn("cdc_datetime_updated", current_timestamp())


@node(
    type="pyspark",
    dependencies=[t2],
    cache=IncrementalCDCCache(
        spark=spark,
        table="customer_data_final",
        merge_keys=["id"],
        schema="default",
        cdc_table="cdc_metadata"
    )
)
def t3(t2):
    """
    Final node with CDC cache.
    Adds business logic and maintains timestamp.
    """
    return t2.selectExpr(
        "id",
        "name",
        "processed_name",
        "id * 100 as customer_score",
        "cdc_datetime_updated"
    )


## Setup - Clean Environment

Before running our example, let's clean up any existing tables.


In [ ]:
# Clean up any existing tables
spark.sql("DROP TABLE IF EXISTS default.customer_data_processed")
spark.sql("DROP TABLE IF EXISTS default.customer_data_final")
spark.sql("DROP TABLE IF EXISTS default.cdc_metadata")

print("✓ Environment cleaned")


## First Run - Initial Data Load

Process the initial batch of customer data (customers 1 and 2).

We use `CacheMode.MERGE` to enable incremental appending to the cache tables.


In [ ]:
from flypipe.cache import CacheMode

# First batch of customer data
initial_input = spark.createDataFrame(
    data=[(1, "Alice"), (2, "Bob")],
    schema=["id", "name"]
)

print("=" * 60)
print("FIRST RUN - Processing initial data (customers 1, 2)")
print("=" * 60)

result1 = t3.run(
    spark,
    inputs={t1: initial_input},
    parallel=False,
    cache={t2: CacheMode.MERGE, t3: CacheMode.MERGE}
)

print("\n📊 Result of first run:")
result1.show()

print("\n📦 Cached data in final table:")
spark.table("default.customer_data_final").show()


## Second Run - Incremental Load (CDC in Action!)

Now we'll process a new batch with only new customers (3 and 4).

The CDC cache will:
1. Filter the data based on timestamps
2. Process only the new rows
3. Append results to the existing cache tables


In [ ]:
# Second batch - only NEW customers
new_input = spark.createDataFrame(
    data=[(3, "Charlie"), (4, "Diana")],  # Only new rows
    schema=["id", "name"]
)

print("=" * 60)
print("SECOND RUN - Processing new data (customers 3, 4)")
print("=" * 60)

result2 = t3.run(
    spark,
    inputs={t1: new_input},
    parallel=False,
    cache={t2: CacheMode.MERGE, t3: CacheMode.MERGE}
)

print("\n📊 Result of second run (only new rows processed):")
result2.show()

print("\n📦 Full cached data in final table (all historical data):")
spark.table("default.customer_data_final").orderBy("id").show()


## Viewing CDC Metadata

The CDC manager tracks when each edge in the pipeline was last processed.


In [ ]:
print("📋 CDC Metadata Table:")
print("Tracks when each pipeline edge was last processed\n")
spark.table("default.cdc_metadata").orderBy("source", "destiny", "cdc_datetime_updated").show(truncate=False)


## How It Works

### First Run (Initial Load)
- **Input**: Customers 1 and 2
- **Processing**: All nodes execute transformations
- **Caching**: Results written to Delta tables with timestamps (CREATE TABLE)
- **CDC Metadata**: Records when t1→t2 and t2→t3 edges were processed

### Second Run (Incremental Load)
- **Input**: Customers 3 and 4 (NEW data only)
- **CDC Filtering**: 
  - `read_cdc()` checks timestamps from CDC metadata
  - Returns only rows with `cdc_datetime_updated` > last processed time
  - In this case, all rows are new (no filtering needed)
- **Processing**: Only new customers are transformed
- **Caching**: Results merged into existing tables using Delta Lake's MERGE INTO
  - Uses `merge_keys` (e.g., `id`) to match existing records
  - WHEN MATCHED: Updates existing records
  - WHEN NOT MATCHED: Inserts new records
- **Final State**: Cache contains all 4 customers (1, 2, 3, 4)

### Benefits

✅ **Performance**: Only process new/changed data  
✅ **Storage**: Maintain complete historical data in cache  
✅ **Scalability**: Handle large datasets incrementally  
✅ **Auditability**: Track when each data segment was processed  
✅ **Upsert Logic**: Automatically handles inserts and updates with MERGE INTO


## Advanced: Querying CDC Metadata

You can query the CDC metadata to understand your pipeline's processing history.


In [ ]:
# Find the last processing time for a specific edge
query = """
    SELECT 
        source,
        destiny,
        MAX(cdc_datetime_updated) as last_processed,
        COUNT(*) as run_count
    FROM default.cdc_metadata
    GROUP BY source, destiny
    ORDER BY source, destiny
"""

print("📊 CDC Processing Summary:")
spark.sql(query).show(truncate=False)


## Production Use Cases

### Example 1: Daily Batch Processing

```python
# Day 1: Process initial load
initial_customers = spark.read.parquet("s3://data/customers/2024-01-01/")
pipeline.run(spark, inputs={source_node: initial_customers}, cache={...CacheMode.MERGE})

# Day 2: Process only new/changed customers
new_customers = spark.read.parquet("s3://data/customers/2024-01-02/")
pipeline.run(spark, inputs={source_node: new_customers}, cache={...CacheMode.MERGE})
# CDC automatically filters and processes only new rows
```

### Example 2: Streaming Data Processing

```python
# Micro-batch 1 (10:00 AM)
batch1 = spark.read.format("kafka")...
pipeline.run(spark, inputs={source: batch1}, cache={...CacheMode.MERGE})

# Micro-batch 2 (10:05 AM) - only new events
batch2 = spark.read.format("kafka")...
pipeline.run(spark, inputs={source: batch2}, cache={...CacheMode.MERGE})
```

### Example 3: Data Warehouse Updates

```python
# Weekly dimension table update
# CDC ensures only new/changed dimension records are processed
# and merged into the existing warehouse table
```


## Summary

CDC Cache in Flypipe enables efficient incremental data processing by:

1. **Tracking timestamps** - CDC metadata table records when each pipeline edge was processed
2. **Filtering data** - `read_cdc()` returns only rows updated after the last processing timestamp
3. **Appending results** - `CacheMode.MERGE` appends new results to existing cache tables
4. **Maintaining history** - Full dataset is preserved in cache while processing only deltas

### Key Components

- **CDCManager**: Manages CDC metadata in a Spark Delta table (created internally by the cache)
- **IncrementalCDCCache**: Extends `CDCCache` with `read_cdc()` and `write_cdc()` methods
  - Accepts `spark`, `table`, `merge_keys`, `schema`, `catalog`, and `cdc_table` parameters
  - Uses Delta Lake's MERGE INTO for upsert operations based on `merge_keys`
  - Creates `CDCManager` internally in the constructor
- **cdc_datetime_updated**: Timestamp column added to track row processing time
- **CacheMode.MERGE**: Cache mode that triggers MERGE INTO for incremental updates

This pattern is ideal for large-scale data pipelines where reprocessing all data would be inefficient or expensive.
